In [1]:
# importing the dependencies
import re
import string
import numpy as np
import pandas as pd

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [5]:
#laoding the dataset
df=pd.read_csv('/content/twitter.csv')
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [6]:
#keeping only relevant columns
df=df[['tweet','class']].dropna().reset_index(drop=True)


In [7]:
#map numeric class -> human readable labels
class_map={0:"Hate Speech",1:"Offensive Language",2:"No Hate and Offensive"}
df["labels"]=df["class"].map(class_map)


In [8]:
#show dataset overview
print("class distribution:")
print(df["labels"].value_counts())
df.head()

class distribution:
labels
Offensive Language       19190
No Hate and Offensive     4163
Hate Speech               1430
Name: count, dtype: int64


,tweet,class,labels
0,!!! RT @mayasolovely: As a woman you shouldn't...,2,No Hate and Offensive
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,1,Offensive Language
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,1,Offensive Language
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,1,Offensive Language
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,1,Offensive Language


In [9]:
#preprocess tweets
sw=set(stopwords.words("english"))
lem=WordNetLemmatizer()

In [10]:
def clean(text,lemmatize=True):
  text=str(text).lower()
  text=re.sub(r'\brt\b',' ',text)
  text=re.sub(r'@\w+',' ',text)
  text=re.sub(r'https?://\S+\www\.\S+',' ',text)
  text=re.sub(r'<.*?>',' ',text)
  text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)
  text = re.sub(r'\d+', ' ', text)
  text = re.sub(r'\s+', ' ', text).strip()
  tokens = [w for w in text.split() if w not in sw]
  if lemmatize:
    tokens=[lem.lemmatize(w) for w in tokens]
  return " ".join(tokens)

df["clean_tweet"]=df["tweet"].apply(clean)
df[["tweet","clean_tweet","labels"]].head()


,tweet,clean_tweet,labels
0,!!! RT @mayasolovely: As a woman you shouldn't...,woman complain cleaning house amp man always t...,No Hate and Offensive
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,boy dat cold tyga dwn bad cuffin dat hoe st place,Offensive Language
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,dawg ever fuck bitch start cry confused shit,Offensive Language
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,look like tranny,Offensive Language
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,shit hear might true might faker bitch told ya,Offensive Language


In [11]:
#train test split

x=df["clean_tweet"].values
y=df["labels"].values

#encode labels
le=LabelEncoder()
y_enc=le.fit_transform(y)

x_train,x_test,y_train,y_test=train_test_split(
    x,y_enc,test_size=0.33,random_state=42,stratify=y_enc
)

print("Train size:",len(x_train),"| Test size:",len(x_test))

Train size: 16604 | Test size: 8179


In [12]:
#build pipeline(tf-idf + logistic regression)

pipeline=Pipeline([
    ("tfidf",TfidfVectorizer(max_features=20000,ngram_range=(1,2))),
    ("clf",LogisticRegression(max_iter=1000,class_weight="balanced",solver="saga"))
])

pipeline.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    solver='saga'))])

In [13]:
#evaluate model
y_pred=pipeline.predict(x_test)

print("Classification Report:\n")
print(classification_report(y_test,y_pred,target_names=le.classes_))

Classification Report:

                       precision    recall  f1-score   support

          Hate Speech       0.12      0.86      0.20       472
No Hate and Offensive       0.71      0.87      0.78      1374
   Offensive Language       0.99      0.47      0.63      6333

             accuracy                           0.56      8179
            macro avg       0.60      0.73      0.54      8179
         weighted avg       0.89      0.56      0.63      8179



In [14]:
#confusion matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test,y_pred))

Confusion Matrix:

[[ 404   39   29]
 [ 167 1197   10]
 [2921  455 2957]]


In [15]:
#accuracy score
print("Accuracy Score:\n")
print(accuracy_score(y_test,y_pred))

Accuracy Score:

0.5572808411786282


In [16]:
#save model and label encoder
joblib.dump(pipeline,"hate_detector_pipeline.joblib")
joblib.dump(le,"label_encoder.joblib")

print("SAved model")

SAved model


In [17]:
#try out

def predict_text(text):
  clean_text=clean(text)
  probs=pipeline.predict_proba([clean_text])[0]
  pred_idx=np.argmax(probs)
  pred_label=le.inverse_transform([pred_idx])[0]

  return pred_label,{lbl:float(p) for lbl,p in zip(le.classes_,probs)}

#example
sample="I hate you"
label,probas=predict_text(sample)
print("Text:", sample)
print("Prediction:", label)
print("Probabilities:", probas)



Text: I hate you
Prediction: Hate Speech
Probabilities: {'Hate Speech': 0.9883975990550738, 'No Hate and Offensive': 0.010324192532347697, 'Offensive Language': 0.0012782084125785276}
